# 01 · Data Preprocessing
Parse WhatsApp exports → clean → anonymise → save.

In [ ]:
import os, re
import pandas as pd

DATA_PATH = "../data/raw/"   # folder with chat*.txt files

all_chats = ""
for f in os.listdir(DATA_PATH):
    if f.endswith(".txt"):
        with open(os.path.join(DATA_PATH, f), encoding="utf-8") as fh:
            all_chats += fh.read() + "\n"

print(f"Loaded {len(all_chats):,} characters from {DATA_PATH}")

In [ ]:
PATTERN = r"(\d{1,2}/\d{1,2}/\d{2,4}),\s(\d{1,2}:\d{2}\s?[apAPmM]*)\s-\s(.*?): (.*)"
matches = re.findall(PATTERN, all_chats)
df = pd.DataFrame(matches, columns=["date","time","user","message"])
print(f"Total messages parsed: {df.shape[0]}")
df.head()

In [ ]:
# Remove system messages
SYSTEM = ["<Media omitted>","deleted","end-to-end encrypted","added","removed",
          "left","joined","changed","created","pinned"]
for kw in SYSTEM:
    df = df[~df["message"].str.contains(kw, case=False, na=False)]
df = df[df["message"].str.strip() != ""].reset_index(drop=True)
print(f"After cleaning: {df.shape[0]} messages")

In [ ]:
# Anonymise users
unique_users = df["user"].unique()
user_map = {u: f"User{i+1}" for i,u in enumerate(unique_users)}
df["user"] = df["user"].map(user_map)
df.to_csv("../data/whatsapp_cleaned_data.csv", index=False)
print("Saved → data/whatsapp_cleaned_data.csv")
df.head()